# Google Cloud Monitoring Metrics & ADK Telemetry Verification

This notebook demonstrates the initialization and live execution of Google Cloud Monitoring metrics collection using Google ADK's native telemetry framework.

### Verification Highlights:
1. **Provider Initialization**: Verifies `PeriodicExportingMetricReader` and `CloudMonitoringMetricsExporter` setup via `setup_observability()`.
2. **Agent Turn Latency**: Emits `gen_ai.agent.invocation.duration` and increments `gen_ai.agent.invocation.requests`.
3. **Model Operations & Tokens**: Records `gen_ai.client.operation.duration` and `gen_ai.client.token.usage` using `LlmResponse.usage_metadata`.
4. **Tool Latency & Counts**: Records `gen_ai.tool.execution.duration` and `gen_ai.tool.execution.requests`.
5. **Edge Cases & Failure Modes**: Validates non-crashing telemetry during missing metadata or simulated Vertex AI API errors.

In [ ]:
import os
import sys
import time
from unittest.mock import MagicMock

# Append repository root (2 levels up from notebooks/observability/)
sys.path.append("../..")

from opentelemetry import metrics
from google.genai import types
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from agent.core_agent.observability import setup_observability
from agent.core_agent.plugins.observability_plugin.plugin import ObservabilityPlugin
from agent.core_agent.plugins.observability_plugin.metrics import (
    agent_invocation_requests,
    agent_invocation_errors,
    tool_execution_requests,
    tool_execution_errors,
)

print("✓ Imports succeeded successfully.")

## Step 1: Initialize Observability (Logging & Cloud Monitoring)

In [ ]:
setup_observability()
provider = metrics.get_meter_provider()
print(f"✓ Active OpenTelemetry MeterProvider: {type(provider).__name__}")

## Step 2: Instantiate ObservabilityPlugin

In [ ]:
plugin = ObservabilityPlugin()
print(f"✓ ObservabilityPlugin initialized: {plugin.name}")

## Step 3: Verify Agent Turn Metrics (`before_run` and `after_run`)

In [ ]:
mock_turn_ctx = MagicMock()
mock_turn_ctx.session.id = "session-verify-001"
mock_turn_ctx.user_id = "analyst@example.com"
mock_turn_ctx.agent.name = "osiris-coordinator"
mock_turn_ctx.invocation_id = "turn-verify-001"

# Start turn
await plugin.before_run_callback(invocation_context=mock_turn_ctx)
print("✓ Turn started. Start timestamp recorded in plugin state.")

time.sleep(0.15)

# End turn
await plugin.after_run_callback(invocation_context=mock_turn_ctx)
print("✓ Turn completed. gen_ai.agent.invocation.duration emitted.")

## Step 4: Verify Model Latency and Token Usage Metrics

In [ ]:
mock_model_ctx = MagicMock()
mock_model_ctx.invocation_id = "model-verify-001"
mock_model_ctx.agent_name = "research_agent"

llm_request = LlmRequest(model="gemini-2.5-flash")
await plugin.before_model_callback(callback_context=mock_model_ctx, llm_request=llm_request)

time.sleep(0.08)

usage = types.GenerateContentResponseUsageMetadata(
    prompt_token_count=230,
    candidates_token_count=75,
    total_token_count=305,
)
llm_response = LlmResponse(
    content=types.Content(parts=[types.Part(text="Verified telemetry response")]),
    usage_metadata=usage,
    model_version="gemini-2.5-flash",
)

await plugin.after_model_callback(callback_context=mock_model_ctx, llm_response=llm_response)
print(f"✓ Model telemetry emitted: prompt_tokens={usage.prompt_token_count}, completion_tokens={usage.candidates_token_count}")

## Step 5: Verify Tool Execution Metrics

In [ ]:
mock_tool = MagicMock()
mock_tool.name = "execute_bigquery_query"

mock_tool_ctx = MagicMock()
mock_tool_ctx.agent_name = "research_agent"

await plugin.before_tool_callback(tool=mock_tool, tool_args={"query": "SELECT 1"}, tool_context=mock_tool_ctx)
time.sleep(0.04)
await plugin.after_tool_callback(
    tool=mock_tool,
    tool_args={"query": "SELECT 1"},
    tool_context=mock_tool_ctx,
    result={"rows": 1},
)
print("✓ Tool execution metric gen_ai.tool.execution.duration recorded.")

## Step 6: Verify Edge Case & Failure Mode Resilience

In [ ]:
# Edge Case: Missing usage_metadata
empty_usage_response = LlmResponse(
    content=types.Content(parts=[types.Part(text="Streaming chunk")]),
    usage_metadata=None,
    model_version="gemini-2.5-flash",
)
await plugin.after_model_callback(callback_context=mock_model_ctx, llm_response=empty_usage_response)
print("✓ Handled missing usage_metadata without failure.")

# Failure Mode: Model error callback
simulated_error = RuntimeError("Vertex AI transient 503 error")
await plugin.on_model_error_callback(
    callback_context=mock_model_ctx,
    llm_request=llm_request,
    error=simulated_error,
)
print("✓ Model error callback recorded error duration and incremented error counter.")